In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

# --- 1. Chargement et Séparation des Données d'Entraînement ---
try:
    df_train = pd.read_csv('train.csv')
    X = df_train.drop(columns=['id', 'loan_paid_back'])
    y = df_train['loan_paid_back']
    print(f"Chargement de 'train.csv' réussi. Nombre d'échantillons d'entraînement : {len(X)}")
except FileNotFoundError:
    print("Erreur: Le fichier 'train.csv' est introuvable. Veuillez vérifier le chemin.")
    exit() # Arrête le script si les données d'entraînement ne sont pas là.


# --- 2. Définition des Stratégies de Prétraitement ---

# A. Variables Numériques (Standardisation)
numeric_features = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
numeric_transformer = StandardScaler()

# B. Variables Catégorielles Nominales (One-Hot Encoding)
categorical_features = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# C. Variable Catégorielle Ordinale (Ordinal Encoding avec ordre défini)
ordinal_features = ['grade_subgrade']
# On définit l'ordre exact, du plus petit grade au plus grand
grade_order = sorted(df_train['grade_subgrade'].unique())
ordinal_transformer = OrdinalEncoder(categories=[grade_order], handle_unknown='use_encoded_value', unknown_value=-1)

# --- 3. Assemblage du Processeur (ColumTransformer) ---
# Il applique les transformations spécifiques à chaque type de colonne.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('ord', ordinal_transformer, ordinal_features)
    ],
    remainder='passthrough' # Conserve toutes les autres colonnes non listées (s'il y en avait).
)

# --- 4. Le Pipeline Final (Prétraitement + Modèle Gradient Boosting) ---
# Le pipeline garantit la séquence d'opérations et évite le data leakage.
model_gb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, subsample=0.7, random_state=42))
])

# --- 5. Entraînement du modèle final sur TOUTES les données d'entraînement ---
print("\n--- Démarrage de l'Entraînement du Modèle ---")
model_gb.fit(X, y)
print("✅ Entraînement terminé.")


# --------------------------------------------------------------------------------------------------
# --- 6. Prédiction sur le Fichier de Test et Génération de la Soumission ---
# --------------------------------------------------------------------------------------------------

try:
    df_test = pd.read_csv('test.csv')
    print(f"Chargement de 'test.csv' réussi. Nombre d'échantillons de test : {len(df_test)}")
except FileNotFoundError:
    print("\nErreur: Le fichier 'test.csv' est introuvable. Impossible de générer la soumission.")
    exit()

if not df_test.empty:
    # A. Isoler les features du test
    test_ids = df_test['id']
    X_test = df_test.drop(columns=['id'])

    # B. Générer les Probabilités
    # Le pipeline applique automatiquement le StandardScaler et l'OneHotEncoder appris sur df_train à X_test.
    print("Génération des prédictions (probabilités de 'loan_paid_back')...")
    # predict_proba retourne un tableau de [probabilité de 0, probabilité de 1]. Nous prenons la colonne 1.
    test_probs = model_gb.predict_proba(X_test)[:, 1]

    # C. Créer le Fichier de Soumission
    submission = pd.DataFrame({
        'id': test_ids,
        'loan_paid_back': test_probs
    })

    # D. Sauvegarde au format requis (sans l'index)
    submission.to_csv('submission.csv', index=False)
    print("\n--- Soumission Générée ---")
    print(f"✅ Fichier 'submission.csv' généré avec succès dans le répertoire courant.")
    print(f"Structure de la soumission (premières lignes) :\n{submission.head()}")

Chargement de 'train.csv' réussi. Nombre d'échantillons d'entraînement : 593994

--- Démarrage de l'Entraînement du Modèle ---
✅ Entraînement terminé.
Chargement de 'test.csv' réussi. Nombre d'échantillons de test : 254569
Génération des prédictions (probabilités de 'loan_paid_back')...

--- Soumission Générée ---
✅ Fichier 'submission.csv' généré avec succès dans le répertoire courant.
Structure de la soumission (premières lignes) :
       id  loan_paid_back
0  593994        0.920424
1  593995        0.981087
2  593996        0.451939
3  593997        0.933430
4  593998        0.959159
